## TIFON Databases. Read and load example databases

In [ ]:
try:
    import pyLOM
    print('Environment with pylom instaled')
except ImportError:
    import sys
    sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM
import numpy as np

def read_db(datafolder, case_idx, make_data_dict:bool = True):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    if make_data_dict:
        db.extract_inputs(
            id_groups = (3,),
            cases_idx = case_idx,
            vtu_type='surface',
            verbose=False
            )

        for stage in [0, 1]:
            
            db.extract_outputs(
                id_groups=(3,),
                stage=stage, cases_idx = case_idx,
                var_name_excluded = [
                    'BoundaryValues_CoefSkinFrictionX',
                    'BoundaryValues_CoefSkinFrictionY',
                    'BoundaryValues_CoefSkinFrictionZ'
                    ],
                vtu_type='surface',
                )
    
    return db

In [ ]:
db_1 = read_db('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans5/', case_idx='all', make_data_dict=False)

### Residuals

In [ ]:
db_1.plot_state()

In [ ]:
mon = db_1.residuals.monitor_cases([198,])   # case_idx o nombres de carpeta
print(mon)

In [ ]:
db_1.residuals.plot_all_final_residuals(
    mode="scaled",
    stage=[1,],
    only_finished=True,
    activate_idx = True,
    figsize=(20, 20)
)

In [ ]:
for c in [209, 242, 230, 216, 214]:
    print(c, db_1.case_per_idx(c))

In [ ]:
for c in [197, 203]:
    print(c)
    db_1.residuals.plot_residuals_from_case(
        case_idx = c,
        stage='all',
        mode='scaled',
    )

In [ ]:
db_1.residuals.plot_state_calculation(num_stages=1, txt_from_end=1)

### Calculate df_post from get_df_metrics()

In [ ]:
db_1.residuals.plot_all_final_residuals(mode='scaled', stage=[0,], lim_converged=1e-4, only_finished=True, print_non_converged=False, activate_idx=True)

In [ ]:
db_1.residuals.plot_all_final_residuals(mode='scaled', stage=[0, 1], lim_converged=1e-5, only_finished=True, print_non_converged=False, activate_idx=True)

In [ ]:
all = list(range(80))
to_remove = [0, 3, 4, 6, 7, 9, 13, 15, 16, 19, 25, 28, 31, 39, 42, 45, 50, 54, 60, 63, 69, 72, 75]
print(1-(len(to_remove)/len(all)))

In [ ]:
res_mean, res_std = db_1.residuals.integrals_convergence_criteria(
    iterations_back = 1000,
    only_finished = True,
    only_converged = False, 
    mode = '3D',
    plot=True,
    residual_threshold = 1e-3,
    verbose=True
)

In [ ]:
df_finals = db_1.residuals.get_all_final_residuals(verbose=False, stage=[0,], only_finished=False, load_in_metadata=False)
display(df_finals)

In [ ]:
df_post = db_1.residuals.get_df_metrics(var_metrics = ['CoefLift', 'CoefDrag', 'CoefMomentY'],
    iter_var = 1000,
    save =  True
)

### Analizar df_post

In [1]:
import pandas as pd
df_post = pd.read_csv(
    '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans5/metadata/df_post.csv',
    sep=',',
    index_col=0   
)
df_post['dataset'] = 'dataset_0'
# for stage in [0, 1]:
#     col=f'coeflift_mean_stage{stage}'
#     arr = df_post.loc[df_post['dataset'] == 'dataset_3', col] * 0.1
#     df_post.loc[df_post['dataset'] == 'dataset_3', col] = arr
    
#     col=f'coefdrag_mean_stage{stage}'
#     arr = df_post.loc[df_post['dataset'] == 'dataset_3', col] * 0.1
#     df_post.loc[df_post['dataset'] == 'dataset_3', col] = arr
    
# df_post.to_csv(
#     '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_#280/metadata/df_post.csv',
#     sep=',',
#     index_label = 'index'  
# )
# df_post['size'] = df_post['densityresidual_scaled_stage1'].apply(lambda x: -np.log10(x))
# df_post['size'] = 10*(df_post['size'] - df_post['size'].min()) / (df_post['size'].max() - df_post['size'].min())
# display(df_post[['aoa', 'mach', 'size', 'dataset']])
# print(df_post['size'].min(), df_post['size'].max())

In [22]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from typing import Union

def plot_df_post(df_post, col1:str, col2:str, col3:str = None, line_xy:bool = True, save_path:Union['str', bool] = False):
    
    df_post['size'] = df_post['densityresidual_scaled_stage1'].apply(lambda x: 1/-np.log10(x))
    
    df_post['size'] = 10*(df_post['size'] - df_post['size'].min()) / (df_post['size'].max() - df_post['size'].min())
    dicc_batch = {
        'dataset_0': 'batch_0',
        # 'dataset_1': 'batch_0',
        # 'dataset_2': 'batch_1',
        # 'dataset_3': 'batch_2',
    }
    
    df_post['batch'] = df_post['dataset'].apply(lambda x: dicc_batch[x])
    df_post.columns = [col.lower() for col in df_post.columns.to_list()]
    if col3 is not None:
        fig = px.scatter(
            df_post,
            x=col1, y=col2,
            color=col3,
            symbol="dataset",
            symbol_sequence=["circle", "diamond", "triangle-up", "square"],
            hover_data=df_post.columns,
            #tamaño del marker inversamente proporcional al valor de la columna densityresidual_stage0 
            size=df_post['size']
        )
    else:
        fig = px.scatter(
            df_post, 
            x=col1, y=col2, 
            symbol="dataset",
            symbol_sequence=["circle", "diamond", "triangle-up", "square"],
            hover_data=df_post.columns
            )
    
    if line_xy:
        min_val = min(df_post[col1].min(), df_post[col2].min())
        max_val = max(df_post[col1].max(), df_post[col2].max())
        fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
                      line=dict(color='Red', dash='dash'))
    
    fig.update_layout(
        title=f'{col1} vs {col2}',
        xaxis_title=col1,
        yaxis_title=col2,
        height=500, #
        width=800
        )
    # poner la leyenda a la izquierda
    fig.update_layout(legend=dict(x=0, y=1))
    #guardar figura como png si save_path es un string
    if save_path:
        fig.write_image(save_path)
        
    else:
        fig.show()
        
def plot_physical_curves(df_post, stage):

    df_post = df_post.copy()

    df_post['size'] = df_post['densityresidual_scaled_stage1'].apply(
        lambda x: 1 / -np.log10(x)
    )

    df_post['size'] = 10 * (
        (df_post['size'] - df_post['size'].min()) /
        (df_post['size'].max() - df_post['size'].min())
    )

    dicc_batch = {
        'dataset_0': 'batch_0',
    }

    df_post['batch'] = df_post['dataset'].apply(lambda x: dicc_batch[x])
    df_post.columns = [col.lower() for col in df_post.columns]

    df_post['mach'] = df_post['mach'].round(2)
    df_post = df_post.sort_values(['mach', 'aoa'])

    # Configuración común
    colorscale = 'Viridis'
    mach_values = sorted(df_post['mach'].unique())

    mach_min = min(mach_values)
    mach_max = max(mach_values)

    def add_mach_curves(fig, x_col, y_col, x_title, y_title):

        for mach in mach_values:

            df_mach = df_post[df_post['mach'] == mach]

            color_norm = (
                (mach - mach_min) / (mach_max - mach_min)
                if mach_max != mach_min else 0.5
            )

            color = px.colors.sample_colorscale(
                colorscale, color_norm
            )[0]

            fig.add_trace(
                go.Scatter(
                    x=df_mach[x_col],
                    y=df_mach[y_col],
                    mode='lines',
                    line=dict(color=color, width=2),
                    name=f'Mach {mach:.2f}',
                    showlegend=False,
                    hovertemplate=(
                        f'Mach: {mach:.2f}<br>'
                        'X: %{x:.5f}<br>'
                        'Y: %{y:.5f}'
                        '<extra></extra>'
                    )
                )
            )

        # Colorbar común
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode='markers',
                marker=dict(
                    size=0,
                    color=[mach_min, mach_max],
                    cmin=mach_min,
                    cmax=mach_max,
                    colorscale=colorscale,
                    showscale=True,
                    colorbar=dict(
                        title='Mach',
                        thickness=20,
                        len=0.8,
                    )
                ),
                showlegend=False,
                hoverinfo='skip'
            )
        )

        # Añadir marcadores individuales
        fig_scatter = px.scatter(
            df_post,
            x=x_col,
            y=y_col,
            symbol='dataset',
            symbol_sequence=[
                'circle', 'diamond', 'triangle-up', 'square'
            ],
            hover_data=df_post.columns,
        )

        for trace in fig_scatter.data:
            trace.showlegend = False
            fig.add_trace(trace)

        fig.update_layout(
            xaxis_title=x_title,
            yaxis_title=y_title,
            height=500,
            width=800,
        )

    # --------------------------------------------------
    # 1. CL vs AoA
    # --------------------------------------------------

    fig_cl = go.Figure()

    add_mach_curves(
        fig_cl,
        x_col='aoa',
        y_col=f'coeflift_mean_stage{stage}',
        x_title='AoA',
        y_title='cL',
    )

    fig_cl.update_layout(title='AoA vs cL - Mach constant')
    fig_cl.show()

    # --------------------------------------------------
    # 2. Polar CD vs CL
    # --------------------------------------------------

    fig_polar = go.Figure()

    add_mach_curves(
        fig_polar,
        x_col=f'coeflift_mean_stage{stage}',
        y_col=f'coefdrag_mean_stage{stage}',
        x_title='cL',
        y_title='cD',
    )

    fig_polar.update_layout(title='Aerodynamic polar - Mach constant')
    fig_polar.show()
    
    # --------------------------------------------------
    # 3. CD vs Mach
    # --------------------------------------------------

    # Ancho de las bandas de AoA
    aoa_band_width = 0.5

    df_cd_mach = df_post.copy()

    # Asignar cada simulación a una banda de AoA
    df_cd_mach['aoa_band'] = (
        np.round(df_cd_mach['aoa'] / aoa_band_width) * aoa_band_width
    )

    # Ordenar los puntos por Mach dentro de cada banda
    df_cd_mach = df_cd_mach.sort_values(['aoa_band', 'mach'])

    fig_cd_mach = go.Figure()

    aoa_bands = sorted(df_cd_mach['aoa_band'].unique())

    colors = px.colors.sample_colorscale(
        'Viridis',
        np.linspace(0, 1, len(aoa_bands))
    )

    for aoa_band, color in zip(aoa_bands, colors):

        df_band = df_cd_mach[df_cd_mach['aoa_band'] == aoa_band]

        fig_cd_mach.add_trace(
            go.Scatter(
                x=df_band['mach'],
                y=df_band[f'coefdrag_mean_stage{stage}'],
                mode='lines+markers',
                name=f'AoA ≈ {aoa_band:.1f}°',
                line=dict(color=color, width=2),
                marker=dict(size=5),
                customdata=df_band[['aoa', 'dataset']].values,
                hovertemplate=(
                    'Mach: %{x:.3f}<br>'
                    'cD: %{y:.6f}<br>'
                    'AoA real: %{customdata[0]:.3f}°<br>'
                    'Dataset: %{customdata[1]}'
                    '<extra></extra>'
                )
            )
        )

    fig_cd_mach.update_layout(
        title='Drag coefficient vs Mach number',
        xaxis_title='Mach',
        yaxis_title='cD',
        template='plotly_white',
        legend_title='AoA',
        hovermode='closest'
    )

    fig_cd_mach.show()

In [23]:
plot_physical_curves(df_post, stage='1')

In [24]:
save_path = None # '/home/m.jaraiz/repos/tmp/'
import os
plot_df_post(df_post, col1='coeflift_mean_stage0', col2='coeflift_mean_stage1', col3='batch', line_xy=True, save_path=os.path.join(save_path, 'cl0_vs_cl1.png') if save_path is not None else save_path)
plot_df_post(df_post, col1='coefdrag_mean_stage0', col2='coefdrag_mean_stage1', col3='batch', line_xy=True, save_path=os.path.join(save_path, 'cd0_vs_cd1.png') if save_path is not None else save_path) #cd0_vs_cd1
for stage in [0, 1]:

    plot_df_post(df_post, col1='aoa', col2=f'coeflift_mean_stage{stage}', col3='mach', line_xy=False, save_path=os.path.join(save_path, f'aoa_vs_cl{stage}.png') if save_path is not None else save_path) # aoa_vs_cl{stage}
    plot_df_post(df_post, col1=f'coeflift_mean_stage{stage}', col2=f'coefdrag_mean_stage{stage}', col3='mach', line_xy=False, save_path=os.path.join(save_path, f'cl{stage}_vs_cd{stage}.png') if save_path is not None else save_path) # cl{stage}_vs_cd{stage}
    plot_df_post(df_post, col1='mach', col2=f'coefdrag_mean_stage{stage}', col3='aoa', line_xy=False, save_path=os.path.join(save_path, f'mach_vs_cd{stage}.png') if save_path is not None else save_path) # mach_vs_cd{stage}

In [ ]:

fig = px.scatter(
    df_post,
    x='CoefDrag_mean_stage0',
    y='CoefDrag_mean_stage1',
    color='mach',
    hover_data=['case_idx', 'mach']
)

fig.show()
fig = px.scatter(
    df_post,
    x='CoefLift_mean_stage0',
    y='CoefLift_mean_stage1',
    color='mach',
    hover_data=['case_idx', 'mach']
)
fig.show()
fig = px.scatter(
    df_post,
    x="aoa",
    y="CoefLift_mean_stage1",
    color="mach",
    # title="CL vs AoA"
)
fig.show()
fig = px.scatter(
    df_post,
    x="CoefDrag_mean_stage1",
    y="CoefLift_mean_stage1",
    color="aoa"
)
fig.show()
fig = px.scatter(
    df_post,
    x="CoefDrag_mean_stage1",
    y="CoefLift_mean_stage1",
    color="mach"
)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# -------------------------
# Renderer para VSCode
# -------------------------
pio.renderers.default = "vscode"

# -------------------------
# Crear subplots
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Stage0 vs Stage1 (CL)",
        "Polar CL vs AoA",
        "Drag Polar (CL vs CD)"
    )
)

# -------------------------
# 1. CL Stage0 vs Stage1
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_post['CoefLift_mean_stage0'],
        y=df_post['CoefLift_mean_stage1'],
        mode='markers',
        marker=dict(
            color=df_post['aoa'],
            colorscale='Viridis',
            colorbar=dict(title="AoA"),
            showscale=True
        ),
        text=df_post['case_idx'],
        customdata=df_post[['mach']],
        hovertemplate=
        "CL0=%{x}<br>CL1=%{y}<br>AoA=%{marker.color}<br>Mach=%{customdata[0]}<br>Case=%{text}",
        name="CL stage comparison"
    ),
    row=1, col=1
)

# -------------------------
# 2. Polar CL vs AoA
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_post["aoa"],
        y=df_post["CoefLift_mean_stage1"],
        mode="markers",
        marker=dict(
            color=df_post["mach"],
            colorscale="Viridis",
            showscale=False
        ),
        text=df_post["case_idx"],
        hovertemplate=
        "AoA=%{x}<br>CL=%{y}<br>Mach=%{marker.color}<br>Case=%{text}",
        name="Polar"
    ),
    row=1, col=2
)

# -------------------------
# 3. Drag polar
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_post["CoefDrag_mean_stage1"],
        y=df_post["CoefLift_mean_stage1"],
        mode="markers",
        marker=dict(
            color=df_post["aoa"],
            colorscale="Viridis",
            showscale=False
        ),
        text=df_post["case_idx"],
        customdata=df_post[['mach']],
        hovertemplate=
        "CD=%{x}<br>CL=%{y}<br>AoA=%{marker.color}<br>Mach=%{customdata[0]}<br>Case=%{text}",
        name="Drag polar"
    ),
    row=1, col=3
)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=500,
    width=1400,
    title="CFD Interactive Analysis Dashboard",
)

# -------------------------
# Interactividad extra (selector dataset)
# -------------------------
datasets = df_post['dataset'].unique()

buttons = []
for s in datasets:
    mask = df_post['stage'] == s

    visible = []
    for _ in range(3):  # 3 traces
        visible.append(True)

    buttons.append(dict(
        label=s,
        method="update",
        args=[{
            "x": [
                df_post.loc[mask, 'CoefLift_mean_stage0'],
                df_post.loc[mask, 'aoa'],
                df_post.loc[mask, 'CoefDrag_mean_stage1']
            ],
            "y": [
                df_post.loc[mask, 'CoefLift_mean_stage1'],
                df_post.loc[mask, 'CoefLift_mean_stage1'],
                df_post.loc[mask, 'CoefLift_mean_stage1']
            ],
        }]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.2,
            showactive=True
        )
    ]
)

fig.show()